In [1]:
import io
import os
import random
import torch
import json
import contextlib
import numpy as np
from ultralytics import YOLO 
from mylib import simsettings
from mylib import myutils
from mylib import simtools
from mylib import yolo_patch_softmax as _
from constants import OBS_SCALE, CELL_SIDE, MAP_RESOLUTION, DISPLAY_STEP
from constants import AGENT_HEIGHT, AGENT_RADIUS 
from constants import MAX_ITER, CONFIDENCE_THRESHOLD
from constants import ACTIONS   
from dotenv import load_dotenv
load_dotenv()

import habitat_sim
import habitat_sim.nav as nav
from habitat.utils.visualizations import maps
from habitat_sim.utils import common as utils

# Load YOLO model
yolo_model = YOLO("yolo11x.pt")  

# Initialize cuda:0 device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
with contextlib.redirect_stdout(io.StringIO()):
    yolo_model = yolo_model.to(device)

In [2]:
# Load the JSON file for simulation
with open('simulation-data/simulations.json', 'r') as f:
    simulations = json.load(f)

# Simulation configuration (index selects the simulation)
simulation = simulations[2]

# Load the JSON file for RGB camera intrinsics
with open('simulation-data/camera-intrinsics.json', 'r') as f:
    intrinsics = json.load(f)

In [ ]:
class ObjectSearchEnv:
    def __init__(self, simulation, yolo_model):
        self.simulation = simulation
        self.yolo_model = yolo_model
        self.scene = simulation["scene"]
        self.target_object = simulation["target_object"]
        self.target_object_id = simulation["target_object_id"]
        self.real_target_location = simulation["target_object_location"]

        dataset_config_file = os.path.join(os.getenv("AI2THOR_DATA"), "ai2thor-hab.scene_dataset_config.json")
        sim_settings_dict = {
            "seed": 1,
            "dataset": dataset_config_file,
            "scene": self.scene,
            "width": 1024,
            "height": int(1024 * OBS_SCALE),
            "default_agent": 0,
            "sensor_height": AGENT_HEIGHT,
            "color_sensor": True,
            "depth_sensor": True,
            "enable_physics": False,
        }
        cfg = simsettings.make_cfg(sim_settings_dict)
        self.sim = habitat_sim.Simulator(cfg)
        self.agent = self.sim.initialize_agent(sim_settings_dict["default_agent"])

        self.actions = list(ACTIONS)

        self._setup_maps()

    def _setup_maps(self):
        navmesh_settings = habitat_sim.NavMeshSettings()
        navmesh_settings.agent_height = AGENT_HEIGHT
        navmesh_settings.agent_radius = AGENT_RADIUS
        navmesh_settings.agent_max_climb = 0.2
        navmesh_settings.agent_max_slope = 45.0
        navmesh_settings.include_static_objects = True
        self.sim.recompute_navmesh(self.sim.pathfinder, navmesh_settings)

        self.topdown_map = maps.get_topdown_map(self.sim.pathfinder, height=0, meters_per_pixel=MAP_RESOLUTION, draw_border=True)
        self.topdown_map = myutils.retain_largest_white_chunk(myutils.add_axis_to_map(myutils.map_to_rgb(self.topdown_map)))
        self.topdown_resolution = self.topdown_map.shape[:2]

        self.grid_map = maps.get_topdown_map(self.sim.pathfinder, height=0, meters_per_pixel=CELL_SIDE, draw_border=False)
        self.grid_map = myutils.retain_largest_white_chunk(myutils.map_to_rgb(self.grid_map))
        self.grid_resolution = self.grid_map.shape[:2]

        self.grid_free_cells, self.world_free_coords, self.map_free_cells = [], [], []

        for x_g in range(self.grid_resolution[0]):
            for y_g in range(self.grid_resolution[1]):
                if self.grid_map[x_g, y_g, 0] == 255:
                    real_wrld_z, real_wrld_x = maps.from_grid(x_g, y_g, self.grid_resolution, pathfinder=self.sim.pathfinder)
                    map_x, map_y = maps.to_grid(real_wrld_z, real_wrld_x, self.topdown_resolution, pathfinder=self.sim.pathfinder)

                    if self.sim.pathfinder.is_navigable([real_wrld_x, 0.0, real_wrld_z]) and self.topdown_map[map_x, map_y, 0] == 255:
                        self.grid_free_cells.append([x_g, y_g])
                        self.world_free_coords.append([real_wrld_x, 0.0, real_wrld_z])
                        self.map_free_cells.append([map_x, map_y])
                    else:
                        self.grid_map[x_g, y_g, :] = [128, 128, 128]

    def reset(self):
        self.num_actions = 0
        self.traversed_distance = 0.0
        self.target_found = False

        self.grid_position = random.choice(self.grid_free_cells)
        idx = self.grid_free_cells.index(self.grid_position)
        self.world_position = self.world_free_coords[idx]
        self.map_position = self.map_free_cells[idx]
        self.agent_yaw = random.choice([0, 90, 180, 270])

        agent_state = habitat_sim.AgentState()
        agent_state.position = self.world_position
        agent_state.rotation = myutils.yaw_to_quaternion(self.agent_yaw)
        self.agent.set_state(agent_state)

        self.agent_state = agent_state  # for step()
        return self._get_state()

    def step(self, action):
        if not simtools.is_action_valid(action, self.grid_position, self.agent_yaw, self.grid_free_cells):
            return self._get_state(), -0.01, False, {}

        self.grid_position, self.agent_yaw = simtools.perform_action(action, self.grid_position, self.agent_yaw)
        idx = self.grid_free_cells.index(self.grid_position)
        self.world_position = self.world_free_coords[idx]
        self.map_position = self.map_free_cells[idx]

        self.agent_state.position = self.world_position
        self.agent_state.rotation = myutils.yaw_to_quaternion(self.agent_yaw)
        self.agent.set_state(self.agent_state)

        self.num_actions += 1

        obs = self.sim.get_sensor_observations(0)
        rgb = obs["color_sensor"]
        detections = self._detect_objects(rgb)
        simtools.merge_rgb_yolo_outputs(rgb, detections)
        self.target_found, _ = simtools.was_target_found(self.target_object_id, detections, CONFIDENCE_THRESHOLD)

        reward = 1.0 if self.target_found else -0.01
        done = self.target_found or self.num_actions >= MAX_ITER
        return self._get_state(), reward, done, {}

    def _detect_objects(self, rgb):
        results = self.yolo_model.predict(source=rgb[:, :, :3], device='cuda:0', conf=0.30, iou=0.40, verbose=False, max_det=10)
        return simtools.parse_yolo_detections(results)


    def _get_state(self):
        # --- Agent pose (grid-based) ---
        grid_x, grid_z = self.grid_position
        norm_x = grid_x / self.grid_resolution[0]
        norm_z = grid_z / self.grid_resolution[1]
        norm_yaw = self.agent_yaw / 360.0  # normalize to [0,1]

        pose_vec = np.array([norm_x, norm_z, norm_yaw], dtype=np.float32)

        # --- Local occupancy patch (5x5) ---
        patch = np.zeros((5, 5), dtype=np.float32)
        cx, cz = self.grid_position

        for dx in range(-2, 3):
            for dz in range(-2, 3):
                gx = cx + dx
                gz = cz + dz
                if 0 <= gx < self.grid_resolution[0] and 0 <= gz < self.grid_resolution[1]:
                    is_free = self.grid_map[gx, gz, 0] == 255  # white pixel means free
                    patch[dx + 2, dz + 2] = 1.0 if is_free else 0.0

        patch_vec = patch.flatten()

        # --- Goal embedding (index) ---
        goal_idx = float(self.target_object_id)  # just pass as scalar; use embedding in model

        # --- Concatenate all ---
        state_vec = np.concatenate([pose_vec, patch_vec, [goal_idx]], axis=0)

        return state_vec

In [ ]:
# Load simulation configs and YOLO model
with open('simulation-data/simulations.json', 'r') as f:
    simulations = json.load(f)

simulation = simulations[2]
yolo_model = YOLO("yolo11x.pt").to("cuda:0" if torch.cuda.is_available() else "cpu")

# Create the environment
env = ObjectSearchEnv(simulation, yolo_model)

In [ ]:
# Example episode loop
state = env.reset()
done = False
total_reward = 0.0

while not done:
    action = random.choice(env.actions)
    state, reward, done, info = env.step(action)
    total_reward += reward

print("Episode finished. Total reward:", total_reward)


Renderer: NVIDIA GeForce GTX 1060 6GB/PCIe/SSE2 by NVIDIA Corporation
OpenGL version: 4.6.0 NVIDIA 535.230.02
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_separate_shader_objects
    GL_ARB_robustness
    GL_ARB_texture_storage
    GL_ARB_texture_view
    GL_ARB_framebuffer_no_attachments
    GL_ARB_invalidate_subdata
    GL_ARB_texture_storage_multisample
    GL_ARB_multi_bind
    GL_ARB_direct_state_access
    GL_ARB_get_texture_sub_image
    GL_ARB_texture_filter_anisotropic
    GL_KHR_debug
    GL_KHR_parallel_shader_compile
    GL_NV_depth_buffer_float
Using driver workarounds:
    no-forward-compatible-core-context
    nv-egl-incorrect-gl11-function-pointers
    no-layout-qualifiers-on-old-glsl
    nv-zero-context-profile-mask
    nv-implementation-color-read-format-dsa-broken
    nv-cubemap-inconsistent-compressed-image-size
    nv-cubemap-broken-full-compressed-image-query
    nv-compressed-block-size-in-bits


[15:51:24:082334]:[Warning]:[Metadata] SceneDatasetAttributes.cpp(107)::addNewSceneInstanceToDataset : Dataset : 'ai2thor-hab' : Lighting Layout Attributes '/home/joaocbranco/projects/habitat-meta/ai2thor-hab/ai2thor-hab/configs/scenes/RoboTHOR/FloorPlan_Train3_4.scene_instance.json' specified in Scene Attributes but does not exist in dataset, so creating default.
MeshTools::compile(): ignoring Trade::MeshAttribute::TextureCoordinates 1 as its binding slot is already occupied by Trade::MeshAttribute::TextureCoordinates 0
[15:51:24:791652]:[Warning]:[Sim] Simulator.cpp(595)::instanceStageForSceneAttributes : The active scene does not contain semantic annotations : activeSemanticSceneID_ = 0
Trade::BasisImporter::openData(): missing orientation metadata, assuming Y down. Set the assumeYUp option to suppress this warning.
Trade::BasisImporter::image2D(): Y-flipping a compressed image that's not whole blocks, the result will be shifted by 2 pixels
Trade::BasisImporter::image2D(): Y-flippin

Episode finished. Total reward: -1.5499999999999896
